# 03 - Observability Queries

Consultas de validación sobre el event log del pipeline FinPay.

## Parámetros

In [0]:
dbutils.widgets.text("catalog", "fintech_finpay")
dbutils.widgets.text("schema_observability", "observability")
dbutils.widgets.text("event_log_table", "event_log")

catalog = dbutils.widgets.get("catalog")
schema_observability = dbutils.widgets.get("schema_observability")
event_log_table = dbutils.widgets.get("event_log_table")

event_log_full_name = f"{catalog}.{schema_observability}.{event_log_table}"

print(f"Event Log: {event_log_full_name}")

Event Log: fintech_finpay.observability.event_log


## 1. Validar existencia del Event Log

In [0]:
spark.sql(f"""
SHOW TABLES IN {catalog}.{schema_observability}
""").display()

database,tableName,isTemporary
observability,event_log,false


## 2. Últimos eventos registrados

In [0]:
spark.sql(f"""
SELECT
    timestamp,
    event_type,
    origin.flow_name,
    message
FROM {event_log_full_name}
ORDER BY timestamp DESC
LIMIT 100
""").display()

timestamp,event_type,flow_name,message
2026-05-28T16:04:24.803Z,update_progress,null,Update 351fce is COMPLETED.
2026-05-28T16:04:24.798Z,flow_progress,fintech_finpay.gold.fact_transactions_vm,Reported flow time metrics for flowName: 'fintech_finpay.gold.fact_transactions_vm'.
2026-05-28T16:04:24.795Z,flow_progress,fintech_finpay.silver.quarantine,Reported flow time metrics for flowName: 'fintech_finpay.silver.quarantine'.
2026-05-28T16:04:24.792Z,flow_progress,fintech_finpay.gold.dim_merchant_vm,Reported flow time metrics for flowName: 'fintech_finpay.gold.dim_merchant_vm'.
2026-05-28T16:04:24.789Z,flow_progress,pipelines.flowTimeMetrics.missingFlowName,Reported flow time metrics for flowName: 'pipelines.flowTimeMetrics.missingFlowName'.
2026-05-28T16:04:24.786Z,flow_progress,fintech_finpay.bronze.users,Reported flow time metrics for flowName: 'fintech_finpay.bronze.users'.
2026-05-28T16:04:24.783Z,flow_progress,fintech_finpay.gold.dim_user_vm,Reported flow time metrics for flowName: 'fintech_finpay.gold.dim_user_vm'.
2026-05-28T16:04:24.780Z,flow_progress,fintech_finpay.bronze.stg_users_silver,Reported flow time metrics for flowName: 'fintech_finpay.bronze.stg_users_silver'.
2026-05-28T16:04:24.776Z,flow_progress,fintech_finpay.silver.users,Reported flow time metrics for flowName: 'fintech_finpay.silver.users'.
2026-05-28T16:04:24.773Z,flow_progress,fintech_finpay.silver.merchants,Reported flow time metrics for flowName: 'fintech_finpay.silver.merchants'.


## 3. Eventos por tipo

In [0]:
spark.sql(f"""
SELECT
    event_type,
    COUNT(*) AS total_events
FROM {event_log_full_name}
GROUP BY event_type
ORDER BY total_events DESC
""").display()

event_type,total_events
flow_progress,264
flow_definition,37
dataset_definition,37
update_progress,28
dataset_life_cycle,15
stream_progress,15
create_update,14
runtime_details,12
user_action,11
background_operation,4


## 4. Estado de ejecución por flujo

In [0]:
spark.sql(f"""
SELECT
    origin.flow_name AS flow_name,
    details:flow_progress.status AS status,
    COUNT(*) AS total_events,
    MAX(timestamp) AS last_event_timestamp
FROM {event_log_full_name}
WHERE event_type = 'flow_progress'
GROUP BY
    origin.flow_name,
    details:flow_progress.status
ORDER BY last_event_timestamp DESC
""").display()

flow_name,status,total_events,last_event_timestamp
fintech_finpay.gold.fact_transactions_vm,null,2,2026-05-28T16:04:24.798Z
fintech_finpay.silver.quarantine,null,1,2026-05-28T16:04:24.795Z
fintech_finpay.gold.dim_merchant_vm,null,1,2026-05-28T16:04:24.792Z
pipelines.flowTimeMetrics.missingFlowName,null,13,2026-05-28T16:04:24.789Z
fintech_finpay.bronze.users,null,3,2026-05-28T16:04:24.786Z
fintech_finpay.gold.dim_user_vm,null,1,2026-05-28T16:04:24.783Z
fintech_finpay.bronze.stg_users_silver,null,2,2026-05-28T16:04:24.780Z
fintech_finpay.silver.users,null,1,2026-05-28T16:04:24.776Z
fintech_finpay.silver.merchants,null,1,2026-05-28T16:04:24.773Z
fintech_finpay.gold.dim_user_vm,COMPLETED,1,2026-05-28T16:04:24.662Z


## 5. Registros de entrada y salida por flujo

In [0]:
spark.sql(f"""
SELECT
    origin.flow_name AS flow_name,
    SUM(BIGINT(details:flow_progress.metrics.num_input_rows)) AS input_rows,
    SUM(BIGINT(details:flow_progress.metrics.num_output_rows)) AS output_rows,
    MAX(timestamp) AS last_execution
FROM {event_log_full_name}
WHERE event_type = 'flow_progress'
  AND details:flow_progress.status = 'COMPLETED'
GROUP BY origin.flow_name
ORDER BY last_execution DESC
""").display()

flow_name,input_rows,output_rows,last_execution
fintech_finpay.gold.dim_user_vm,null,null,2026-05-28T16:04:24.662Z
fintech_finpay.silver.quarantine,null,null,2026-05-28T16:04:21.498Z
fintech_finpay.silver.users,null,null,2026-05-28T16:04:20.202Z
fintech_finpay.bronze.stg_users_silver,null,null,2026-05-28T16:04:11.066Z
fintech_finpay.gold.fact_transactions_vm,null,null,2026-05-28T16:04:06.484Z
fintech_finpay.gold.dim_merchant_vm,null,null,2026-05-28T16:03:59.612Z
fintech_finpay.silver.merchants,null,null,2026-05-28T16:03:55.887Z
fintech_finpay.bronze.stg_merchants_silver,null,null,2026-05-28T16:03:43.177Z
fintech_finpay.silver.transactions,null,null,2026-05-28T16:03:17.856Z
fintech_finpay.bronze.merchants,null,null,2026-05-28T16:03:10.946Z


## 6. Calidad de datos por flujo

In [0]:
spark.sql(f"""
SELECT
    origin.flow_name AS flow_name,
    SUM(BIGINT(details:flow_progress.data_quality.dropped_records)) AS dropped_records,
    MAX(timestamp) AS last_execution
FROM {event_log_full_name}
WHERE event_type = 'flow_progress'
  AND details:flow_progress.data_quality IS NOT NULL
GROUP BY origin.flow_name
ORDER BY dropped_records DESC
""").display()

flow_name,dropped_records,last_execution
fintech_finpay.bronze.stg_merchants_silver,0,2026-05-28T16:03:43.160Z
fintech_finpay.gold.dim_merchant_vm,0,2026-05-28T16:03:59.584Z
fintech_finpay.gold.fact_transactions_vm,0,2026-05-28T16:04:06.466Z
fintech_finpay.bronze.stg_users_silver,0,2026-05-28T16:04:11.046Z
fintech_finpay.silver.merchants,0,2026-05-28T16:03:55.862Z
fintech_finpay.bronze.merchants,0,2026-05-28T16:03:10.909Z
fintech_finpay.gold.dim_date_vm,0,2026-05-28T16:01:45.429Z
fintech_finpay.bronze.transactions,0,2026-05-28T16:01:49.359Z
fintech_finpay.silver.transactions,0,2026-05-28T16:03:17.830Z
fintech_finpay.bronze.users,0,2026-05-28T16:03:09.703Z


## 7. Expectativas evaluadas

In [0]:
spark.sql(f"""
SELECT
    origin.flow_name AS flow_name,
    exp.name AS expectation_name,
    SUM(BIGINT(exp.passed_records)) AS passed_records,
    SUM(BIGINT(exp.failed_records)) AS failed_records,
    MAX(timestamp) AS last_execution
FROM {event_log_full_name}
LATERAL VIEW OUTER EXPLODE(
    from_json(
        details:flow_progress.data_quality.expectations,
        'array<struct<name:string,passed_records:bigint,failed_records:bigint>>'
    )
) AS exp
WHERE event_type = 'flow_progress'
  AND details:flow_progress.data_quality IS NOT NULL
GROUP BY
    origin.flow_name,
    exp.name
ORDER BY failed_records DESC
""").display()

flow_name,expectation_name,passed_records,failed_records,last_execution
fintech_finpay.bronze.stg_users_silver,phone_valid_format,0,10230,2026-05-28T16:04:11.046Z
fintech_finpay.bronze.stg_users_silver,email_valid_format,5516,4714,2026-05-28T16:04:11.046Z
fintech_finpay.bronze.stg_transactions_silver,currency_valid_domain,47023,4127,2026-05-28T16:03:00.085Z
fintech_finpay.bronze.stg_transactions_silver,reference_id_rule,47556,3594,2026-05-28T16:03:00.085Z
fintech_finpay.bronze.stg_transactions_silver,channel_valid_domain,48915,2235,2026-05-28T16:03:00.085Z
fintech_finpay.bronze.stg_transactions_silver,status_valid_domain,48992,2158,2026-05-28T16:03:00.085Z
fintech_finpay.bronze.stg_transactions_silver,amount_not_null,49235,1915,2026-05-28T16:03:00.085Z
fintech_finpay.bronze.stg_transactions_silver,amount_positive,49235,1915,2026-05-28T16:03:00.085Z
fintech_finpay.bronze.stg_transactions_silver,transaction_type_valid_domain,50295,855,2026-05-28T16:03:00.085Z
fintech_finpay.bronze.stg_users_silver,segment_valid_domain,9687,543,2026-05-28T16:04:11.046Z


## 8. Tendencia diaria por capa

In [0]:
spark.sql(f"""
SELECT
    DATE(timestamp) AS processing_date,
    CASE
        WHEN origin.flow_name LIKE '%bronze%' THEN 'Bronze'
        WHEN origin.flow_name LIKE '%silver%' OR origin.flow_name LIKE '%stg_%' THEN 'Silver'
        WHEN origin.flow_name LIKE '%gold%' THEN 'Gold'
        ELSE 'Other'
    END AS layer,
    SUM(BIGINT(details:flow_progress.metrics.num_input_rows)) AS input_rows,
    SUM(BIGINT(details:flow_progress.metrics.num_output_rows)) AS output_rows,
    SUM(BIGINT(details:flow_progress.data_quality.dropped_records)) AS dropped_records
FROM {event_log_full_name}
WHERE event_type = 'flow_progress'
GROUP BY
    DATE(timestamp),
    CASE
        WHEN origin.flow_name LIKE '%bronze%' THEN 'Bronze'
        WHEN origin.flow_name LIKE '%silver%' OR origin.flow_name LIKE '%stg_%' THEN 'Silver'
        WHEN origin.flow_name LIKE '%gold%' THEN 'Gold'
        ELSE 'Other'
    END
ORDER BY processing_date DESC
""").display()

processing_date,layer,input_rows,output_rows,dropped_records
2026-05-28,Silver,null,null,0
2026-05-28,Other,null,null,null
2026-05-28,Bronze,null,123760,0
2026-05-28,Gold,null,48652,0


## 9. Tasa de error por tabla

In [0]:
spark.sql(f"""
SELECT
    origin.flow_name AS table_name,
    SUM(BIGINT(details:flow_progress.metrics.num_input_rows)) AS total_input_rows,
    SUM(BIGINT(details:flow_progress.data_quality.dropped_records)) AS total_dropped_records,
    ROUND(
        SUM(BIGINT(details:flow_progress.data_quality.dropped_records)) * 100.0 /
        NULLIF(SUM(BIGINT(details:flow_progress.metrics.num_input_rows)), 0),
        2
    ) AS error_rate_pct
FROM {event_log_full_name}
WHERE event_type = 'flow_progress'
GROUP BY origin.flow_name
ORDER BY error_rate_pct DESC
""").display()

table_name,total_input_rows,total_dropped_records,error_rate_pct
fintech_finpay.bronze.transactions,null,0,null
fintech_finpay.bronze.merchants,null,0,null
fintech_finpay.bronze.stg_merchants_silver,null,0,null
fintech_finpay.gold.dim_date_vm,null,0,null
fintech_finpay.silver.transactions,null,0,null
vw_transactions_silver_prepared,null,null,null
pipelines.flowTimeMetrics.missingFlowName,null,null,null
vw_users_silver_prepared,null,null,null
vw_all_quarantine,null,null,null
fintech_finpay.bronze.users,null,0,null


## 10. Eventos con error

In [0]:
spark.sql(f"""
SELECT
    timestamp,
    event_type,
    origin.flow_name,
    level,
    message,
    error
FROM {event_log_full_name}
WHERE level = 'ERROR'
   OR event_type LIKE '%error%'
ORDER BY timestamp DESC
LIMIT 100
""").display()

timestamp,event_type,flow_name,level,message,error
2026-05-28T15:45:33.502Z,update_progress,null,ERROR,Update e7659c has failed. Failed to analyze flow 'fintech_finpay.bronze.transactions' and 2 other flow(s)..,"List(true, List(List(dlt.py4j_captured_exception.SparkException, Traceback (most recent call last): File ""/Workspace/Users/mmelende@hotmail.com/.bundle/Bundle_20260525/dev/files/src/bronze.py"", cell 1, line 138, in bronze_table dlt.py4j_captured_exception.SparkException: [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/fintech_finpay/default/vol_landing/transactions` is empty. Please try to start the stream when there are files in the input path, or specify the schema. SQLSTATE: 42000, null)))"
2026-05-28T15:45:28.879Z,flow_progress,fintech_finpay.bronze.users,ERROR,Failed to resolve flow: 'fintech_finpay.bronze.users'.,"List(true, List(List(dlt.py4j_captured_exception.SparkException, Traceback (most recent call last): File ""/Workspace/Users/mmelende@hotmail.com/.bundle/Bundle_20260525/dev/files/src/bronze.py"", cell 1, line 138, in bronze_table dlt.py4j_captured_exception.SparkException: [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/fintech_finpay/default/vol_landing/users` is empty. Please try to start the stream when there are files in the input path, or specify the schema. SQLSTATE: 42000, null)))"
2026-05-28T15:45:28.023Z,flow_progress,fintech_finpay.bronze.merchants,ERROR,Failed to resolve flow: 'fintech_finpay.bronze.merchants'.,"List(true, List(List(dlt.py4j_captured_exception.SparkException, Traceback (most recent call last): File ""/Workspace/Users/mmelende@hotmail.com/.bundle/Bundle_20260525/dev/files/src/bronze.py"", cell 1, line 138, in bronze_table dlt.py4j_captured_exception.SparkException: [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/fintech_finpay/default/vol_landing/merchants` is empty. Please try to start the stream when there are files in the input path, or specify the schema. SQLSTATE: 42000, null)))"
2026-05-28T15:45:27.142Z,flow_progress,fintech_finpay.bronze.transactions,ERROR,Failed to resolve flow: 'fintech_finpay.bronze.transactions'.,"List(true, List(List(dlt.py4j_captured_exception.SparkException, Traceback (most recent call last): File ""/Workspace/Users/mmelende@hotmail.com/.bundle/Bundle_20260525/dev/files/src/bronze.py"", cell 1, line 138, in bronze_table dlt.py4j_captured_exception.SparkException: [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/fintech_finpay/default/vol_landing/transactions` is empty. Please try to start the stream when there are files in the input path, or specify the schema. SQLSTATE: 42000, null)))"
2026-05-28T15:44:53.773Z,update_progress,null,ERROR,Update 4ade31 has failed. Failed to analyze flow 'fintech_finpay.bronze.transactions' and 2 other flow(s)..,"List(true, List(List(dlt.py4j_captured_exception.SparkException, Traceback (most recent call last): File ""/Workspace/Users/mmelende@hotmail.com/.bundle/Bundle_20260525/dev/files/src/bronze.py"", cell 1, line 138, in bronze_table dlt.py4j_captured_exception.SparkException: [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/fintech_finpay/default/vol_landing/transactions` is empty. Please try to start the stream when there are files in the input path, or specify the schema. SQLSTATE: 42000, null)))"
2026-05-28T15:44:48.653Z,flow_progress,fintech_finpay.bronze.users,ERROR,Failed to resolve flow: 'fintech_finpay.bronze.users'.,"List(true, List(List(dlt.py4j_captured_exception.SparkException, Traceback (most recent call last): File ""/Workspace/Users/mmelende@hotmail.com/.bundle/Bundle_20260525/dev/files/src/bronze.py"", cell 1, line 138, in bronze_table dlt.py4j_captured_exception.SparkException: [CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE] Cannot infer schema when the input path `/Volumes/fintech_finpay/default/vol_landing/users` is